In [1]:
 
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torch.utils.data import DataLoader,Dataset
import matplotlib.pyplot as plt
import torchvision.utils
import numpy as np
import random
import cv2
from PIL import Image       # PIL (Pillow) is the Python Image Library. Used to cut and resize images, or do simple manipulation.
import torch
from torch.autograd import Variable
import PIL.ImageOps
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

In [30]:
finalClassifierDset = dset.ImageFolder(root='./captured_face_images/',
                                       transform = transforms.Compose([transforms.Grayscale(num_output_channels = 1), transforms.Resize((100,100)), transforms.ToTensor()]))

In [32]:
representation_dataloader = DataLoader(finalClassifierDset, shuffle=False, num_workers=8, batch_size=100)

In [24]:
for i, (_, labels) in enumerate(representation_dataloader):
    print(f"batch {i} labels:", labels.tolist())

batch 0 labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [5]:
class Siamese(torch.nn.Module):
    def __init__(self):
        super(Siamese, self).__init__()
        self.cnn1 = nn.Sequential(
            nn.ReflectionPad2d(1),       #Pads the input tensor using the reflection of the input boundary, it similar to the padding.
            nn.Conv2d(1, 4, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(4),

            nn.ReflectionPad2d(1),
            nn.Conv2d(4, 8, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(8),


            nn.ReflectionPad2d(1),
            nn.Conv2d(8, 8, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(8),
        )

        self.fc1 = nn.Sequential(
            nn.Linear(8*100*100, 500),
            nn.ReLU(inplace=True),

            nn.Linear(500, 500),
            nn.ReLU(inplace=True),

            nn.Linear(500, 5))
        
    def forward(self, x):
        x = self.cnn1(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

In [16]:
myModel = Siamese()
model = torch.load(  'siamese_model.t7', map_location=torch.device('cpu'))
myModel.load_state_dict(model['net_dict'])

<All keys matched successfully>

In [22]:

class SiameseNetworkDataset(Dataset):
    def __init__(self,imageFolderDataset, transform=None):
        self.imageFolderDataset = imageFolderDataset
        self.transform = transform

    # Overriding the data retriever (__getitem__) to provide a pair of images + similar/dissimilar label
    def __getitem__(self,index):
        img0_tuple = random.choice(self.imageFolderDataset.imgs)
        #we need to make sure approx 50% of images are in the same class
        should_get_same_class = random.randint(0,1)
        if should_get_same_class:
            while True:
                #keep looping till the same class image is found
                img1_tuple = random.choice(self.imageFolderDataset.imgs)
                if img0_tuple[1]==img1_tuple[1]:
                    break
        else:
            while True:
                #keep looping till a different class image is found
                img1_tuple = random.choice(self.imageFolderDataset.imgs)
                if img0_tuple[1] !=img1_tuple[1]:
                    break

        img0 = Image.open(img0_tuple[0])
        img1 = Image.open(img1_tuple[0])
        img0 = img0.convert("L")
        img1 = img1.convert("L")

        if self.transform is not None:
            img0 = self.transform(img0)
            img1 = self.transform(img1)

        return img0, img1 , torch.from_numpy(np.array([int(img1_tuple[1]!=img0_tuple[1])],dtype=np.float32))

    def __len__(self):
        return len(self.imageFolderDataset.imgs)

In [33]:
# Get a siamese representation of each of your data points i.e. for each of your team images.
## For example (if your image is of the size 100*100 above)
# <YOUR CODE HERE>
# siamese_dataset = SiameseNetworkDataset(imageFolderDataset = finalClassifierDset,
#                                         transform = transforms.Compose([transforms.Resize((100,100)), transforms.ToTensor()]))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# We will store the feature vectors (X) and labels (y) in these lists
feature_vectors = []
labels_list = []

print("Extracting features from team images...")

# Loop through the 'representation_dataloader' (defined in the notebook)
# 'no_grad()' tells PyTorch we are not training, which saves memory
with torch.no_grad():
    for i, data in enumerate(representation_dataloader, 0):
        images, labels = data
        print("labels:", labels)

        # Move images to the device (GPU/CPU)
        images = images.to(device)

        # Pass images through the model's 'forward_once' method
        # to get the 5-dimensional feature vector
        output = myModel.forward(images)

        print(f"output: {output}")
        # Move the output vectors to the CPU and convert to NumPy arrays
        feature_vectors.append(output.cpu().numpy())
        print(f"labels: {labels}")
        # Store the corresponding labels
        labels_list.append(labels.numpy())

# Concatenate all batches into single NumPy arrays
X_features = np.concatenate(feature_vectors)
y_labels = np.concatenate(labels_list)

print(f"Feature extraction complete.")
print(f"Shape of feature vectors (X): {X_features.shape}")
print(f"Shape of labels (y): {y_labels.shape}")


Extracting features from team images...
labels: tensor([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6])
output: tensor([[-0.4584,  1.4702,  1.8276, -2.2911, -4.7949],
        [-0.9670,  0.1736,  0.6474, -0.6153, -2.0378],
        [-1.3561,  1.2615,  0.1217,  0.2498,  1.5260],
        [-1.5320, -0.0358,  2.2964, -0.6377,  1.7075],
        [-1.2887, -1.8512, -1.4223, -0.3184, -2.5115],
        [-0.9629, -0.3984, -0.1536,  0.3019, -0.5612],
        [ 1.3423,  2.0583, -3.9040, -3.1887, -3.5951],
        [-0.7812,  0.6011, -0.2347, -1.5561, -1.7872],
        [-0.4240, -0.3931, -1.2346, -1.4962, -1.7401],
        [-0.9636,  0.6556, -0.8696,  0.3877, -0.6119],
        [-1.1524, -0.0401,  0.3305, -0.6372,  1.5334],
        [-0.6075,  0.9650,  0.5184, -1.3038, -0.2450],
        [-1.2872,  0.4770,  1.3483, -0.4098,  1.6159],
        [-1.2452,  0.4800,  0.8712, -0.4240,  1.6865]])
labels: tensor([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6])
Feature extraction complete.
Shape of feature vectors (X): (14,

In [ ]:
# YOUR CODE HERE for training a classifier. You can use simple MLP or Sklearn models.
# Note: Ensure you convert torch variable to numpy array before using SkLearn.
print("\nTraining face recognition classifier...")

# (Optional) Split data to check accuracy before final training
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=86, stratify=y_labels
)

# Initialize a Support Vector Classifier (SVC)
# 'probability=True' is needed for the deployment code to work
classifier = SVC(kernel='linear', probability=True)

# Train the classifier
classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = classifier.predict(X_test)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Classifier Accuracy (on test set): {accuracy * 100:.2f}%")

# Now, re-train the classifier on ALL the data for the final deployment model
print("Re-training classifier on all data for final deployment model...")
classifier.fit(X_features, y_labels)

print("Final classifier trained.")

In [ ]:
# YOUR CODE HERE for training a classifier. You can use simple MLP or Sklearn models.
# Note: Ensure you convert torch variable to numpy array before using SkLearn.
print("\nTraining face recognition classifier...")

# (Optional) Split data to check accuracy before final training
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=86, stratify=y_labels
)

# Initialize a Support Vector Classifier (SVC)
# 'probability=True' is needed for the deployment code to work
classifier = SVC(kernel='linear', probability=True)

# Train the classifier
classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = classifier.predict(X_test)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Classifier Accuracy (on test set): {accuracy * 100:.2f}%")

# Now, re-train the classifier on ALL the data for the final deployment model
print("Re-training classifier on all data for final deployment model...")
classifier.fit(X_features, y_labels)

print("Final classifier trained.")